# BirdCLEF+ 2026 — Stage 2 Domain Adaptation Training
V9: GeM + ASL + Soundscape Data Mixing + Background Noise Augmentation

In [ ]:
import subprocess, sys, os

# ── Internet 连通性检查 ──
def _check_internet():
    import urllib.request
    try:
        urllib.request.urlopen('https://huggingface.co', timeout=5)
        return True
    except Exception:
        return False

HAS_INTERNET = _check_internet()
if HAS_INTERNET:
    print('✓ Internet available — pretrained weights will download normally')
else:
    print('✗ Internet NOT available — will use cached/local weights')
    print('  TIP: Kaggle → Settings → Internet → Turn ON for best results')

# 在 import torch 之前检测 GPU，确保安装正确的 PyTorch 版本
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                       capture_output=True, text=True, timeout=10)
    gpu_caps = [float(x.strip()) for x in r.stdout.strip().split('\n') if x.strip()]
    min_cap = min(gpu_caps) if gpu_caps else 99.0
    print(f'GPU compute capabilities: {gpu_caps}')
except Exception:
    min_cap = 99.0
    print('nvidia-smi failed, assuming compatible GPU')

if min_cap < 7.0:
    print(f'SM {min_cap} < 7.0 (P100), installing PyTorch 2.5.1+cu124 BEFORE import...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.5.1', 'torchaudio==2.5.1',
        '--index-url', 'https://download.pytorch.org/whl/cu124'])
    print('Compatible PyTorch installed')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
import os, sys, time, ast, random, warnings, glob as globmod
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

N_GPUS = torch.cuda.device_count()
print(f'PyTorch {torch.__version__}, torchaudio {torchaudio.__version__}, CUDA: {torch.cuda.is_available()}, GPUs: {N_GPUS}')
for i in range(N_GPUS):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

USE_AMP = torch.cuda.is_available()
try:
    from torch.amp import autocast, GradScaler
    AMP_DEVICE = 'cuda'
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    AMP_DEVICE = None

In [ ]:
def find_data_dir():
    if not Path('/kaggle/input').exists(): return Path('../data/raw')
    for p in Path('/kaggle/input').iterdir():
        if (p / 'train.csv').exists(): return p
        if p.is_dir():
            for sub in p.iterdir():
                if sub.is_dir() and (sub / 'train.csv').exists(): return sub
    raise FileNotFoundError('train.csv not found')

# ╔══════════════════════════════════════════════════════════╗
# ║  Experiment: B1 (10s Training Alignment)                 ║
# ║  Base: V14/A1 (LB 0.830)                                ║
# ║  Change: CLIP_DURATION 5→10s, BATCH_SIZE 64→32          ║
# ║  Expected: LB 0.85~0.86 (train-inference alignment)     ║
# ║  Ref: DES-002 V2, EXP-001 #8                           ║
# ╚══════════════════════════════════════════════════════════╝
class CFG:
    IS_KAGGLE = Path('/kaggle/input').exists()
    DATA_DIR = find_data_dir()
    OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('../models')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    SR = 32_000; CLIP_DURATION = 10.0; CLIP_SAMPLES = int(SR * CLIP_DURATION)  # B1: 5→10s
    N_FFT = 1024; HOP_LENGTH = 320; N_MELS = 128; FMIN = 50; FMAX = 14000
    NUM_CLASSES = 234; EPOCHS = 10; BATCH_SIZE = 32  # B1: 64→32 (10s doubles memory)
    LR = 1e-3; WEIGHT_DECAY = 1e-4; NUM_WORKERS = 4; SEED = 42
    N_FOLDS = 5; TRAIN_FOLDS = [0]
    SEC_LABEL_W = 0.3
    MIXUP_P = 0.3; MIXUP_ALPHA = 0.15
    TMASK_P = 0.3; TMASK_W = 50; FMASK_P = 0.3; FMASK_W = 20
    NOISE_P = 0.2; NOISE_STD = 0.005; GAIN_P = 0.3; GAIN_RANGE = (-6, 6)

def set_seed(seed=CFG.SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False; torch.backends.cudnn.benchmark = True

set_seed()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Data: {CFG.DATA_DIR}, Device: {DEVICE}')

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv')
taxonomy = pd.read_csv(CFG.DATA_DIR / 'taxonomy.csv')
submission = pd.read_csv(CFG.DATA_DIR / 'sample_submission.csv', nrows=0)
SPECIES_COLS = [c for c in submission.columns if c != 'row_id']
LABEL2IDX = {label: i for i, label in enumerate(SPECIES_COLS)}
TAX_MAP = dict(zip(taxonomy['primary_label'].astype(str), taxonomy['class_name']))
print(f'Samples: {len(train_df)}, Species: {len(SPECIES_COLS)}')
print(f'Columns: {list(train_df.columns)}')
print(f'\nSample rows:')
print(train_df[['filename','primary_label','secondary_labels','author']].head(3))
print(f'\nAudio files example:')
audio_dir = CFG.DATA_DIR / 'train_audio'
sample_files = list(audio_dir.iterdir())[:5]
for f in sample_files:
    print(f'  {f.name} ({f.stat().st_size/1024:.1f}KB)')

In [ ]:
# ===== DIAGNOSTIC: Test audio loading =====
MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
    n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX, power=2.0)
AMP_TO_DB = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)

RESAMPLERS = {}

def load_audio_fast(path, sr=CFG.SR, duration=CFG.CLIP_DURATION, offset=0.0):
    tl = int(sr * duration)
    try:
        waveform, file_sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if file_sr != sr:
            if file_sr not in RESAMPLERS:
                RESAMPLERS[file_sr] = torchaudio.transforms.Resample(file_sr, sr)
            waveform = RESAMPLERS[file_sr](waveform)
        start = int(offset * sr)
        waveform = waveform[:, start:start + tl].squeeze(0)
    except Exception as e:
        print(f'  LOAD FAILED: {path} -> {e}')
        waveform = torch.zeros(tl)
    if waveform.numel() == 0:
        waveform = torch.zeros(tl)
    if waveform.numel() < tl:
        reps = (tl // waveform.numel()) + 1
        waveform = waveform.repeat(reps)[:tl]
    return waveform[:tl]

def audio_to_melspec_fast(waveform):
    mel = AMP_TO_DB(MEL_TRANSFORM(waveform.unsqueeze(0))).squeeze(0)
    return (mel - mel.mean()) / (mel.std() + 1e-6)

# Test on 5 random files
print('=== Audio Loading Test ===')
test_rows = train_df.sample(5, random_state=42)
for _, row in test_rows.iterrows():
    fpath = str(audio_dir / row['filename'])
    audio = load_audio_fast(fpath)
    mel = audio_to_melspec_fast(audio)
    print(f'  {row["filename"]}: audio={audio.shape} min={audio.min():.4f} max={audio.max():.4f} std={audio.std():.4f} | mel={mel.shape} min={mel.min():.2f} max={mel.max():.2f} std={mel.std():.2f}')
    is_silence = audio.abs().max() < 1e-6
    if is_silence:
        print(f'    WARNING: Audio is silence!')

In [ ]:
# ── 声景标签加载 ──
_sc_path = CFG.DATA_DIR / 'train_soundscapes_labels.csv'
sc_labels_df = pd.read_csv(_sc_path) if _sc_path.exists() else None
sc_dir = CFG.DATA_DIR / 'train_soundscapes'
sc_files = sorted(str(p) for p in sc_dir.rglob('*.ogg')) if sc_dir.is_dir() else []
print(f'Soundscape labels: {len(sc_labels_df) if sc_labels_df is not None else 0} rows, files: {len(sc_files)}')

def _parse_sc_labels(row):
    t = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
    raw = row.get('primary_label', '')
    if pd.isna(raw): return t
    for s in str(raw).split(';'):
        s = s.strip()
        if s in LABEL2IDX: t[LABEL2IDX[s]] = 1.0
    return t

def _sc_offset(row):
    if 'end_time' in row and pd.notna(row.get('end_time')):
        return max(0.0, float(row['end_time']) - CFG.CLIP_DURATION)
    if 'start_time' in row and pd.notna(row.get('start_time')):
        return float(row['start_time'])
    return 0.0

def _mix_snr(clean, noise, snr_db):
    ps = clean.pow(2).mean().clamp(min=1e-10)
    pn = noise.pow(2).mean().clamp(min=1e-10)
    a = torch.sqrt(ps / (10 ** (snr_db / 10.0) * pn + 1e-10))
    return clean + a * noise


class BirdDataset(Dataset):
    def __init__(self, df, is_train=True, sc_ratio=0.3, sc_weight=0.8, bg_noise_p=0.4):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.audio_dir = CFG.DATA_DIR / 'train_audio'
        self.sc_ratio = sc_ratio
        self.sc_weight = sc_weight
        self.bg_noise_p = bg_noise_p

    def __len__(self): return len(self.df)

    def _apply_aug(self, audio):
        if not self.is_train: return audio
        if self.is_train and sc_files and np.random.rand() < self.bg_noise_p:
            nf = random.choice(sc_files)
            noise = load_audio_fast(nf, offset=np.random.uniform(0, 55.0))
            snr = np.random.uniform(5.0, 30.0)
            audio = _mix_snr(audio, noise, snr)
        if np.random.rand() < CFG.NOISE_P:
            audio = audio + torch.randn_like(audio) * CFG.NOISE_STD
        if np.random.rand() < CFG.GAIN_P:
            audio = audio * (10 ** (np.random.uniform(*CFG.GAIN_RANGE) / 20))
        return audio

    def _apply_specaug(self, mel):
        if not self.is_train: return mel
        nm, nf = mel.shape
        if np.random.rand() < CFG.TMASK_P:
            tw = np.random.randint(0, min(CFG.TMASK_W, nf))
            t0 = np.random.randint(0, max(1, nf - tw))
            mel[:, t0:t0+tw] = 0
        if np.random.rand() < CFG.FMASK_P:
            fw = np.random.randint(0, min(CFG.FMASK_W, nm))
            f0 = np.random.randint(0, max(1, nm - fw))
            mel[f0:f0+fw, :] = 0
        return mel

    def _get_soundscape(self):
        j = np.random.randint(0, len(sc_labels_df))
        row = sc_labels_df.iloc[j]
        fn = str(row.get('filename', ''))
        if not fn.endswith('.ogg'): fn += '.ogg'
        offset = _sc_offset(row)
        audio = load_audio_fast(str(sc_dir / fn), offset=offset)
        audio = self._apply_aug(audio)
        mel = self._apply_specaug(audio_to_melspec_fast(audio))
        target = _parse_sc_labels(row)
        return mel.unsqueeze(0), target, torch.tensor(self.sc_weight)

    def __getitem__(self, idx):
        use_sc = (self.is_train and sc_labels_df is not None
                  and len(sc_labels_df) > 0 and np.random.rand() < self.sc_ratio)
        if use_sc:
            return self._get_soundscape()

        row = self.df.iloc[idx]
        offset = np.random.uniform(0, 25.0) if self.is_train else 0.0
        audio = load_audio_fast(str(self.audio_dir / row['filename']), offset=offset)
        audio = self._apply_aug(audio)
        mel = self._apply_specaug(audio_to_melspec_fast(audio))

        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        p = str(row['primary_label'])
        if p in LABEL2IDX: target[LABEL2IDX[p]] = 1.0
        sr = row.get('secondary_labels', '[]')
        if isinstance(sr, str) and sr not in ('[]', ''):
            try:
                for s in ast.literal_eval(sr):
                    if str(s) in LABEL2IDX: target[LABEL2IDX[str(s)]] = CFG.SEC_LABEL_W
            except: pass

        return mel.unsqueeze(0), target, torch.tensor(1.0)

def mixup(mel, tgt, alpha=CFG.MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    p = torch.randperm(mel.size(0))
    return lam*mel+(1-lam)*mel[p], lam*tgt+(1-lam)*tgt[p]

In [ ]:
def _create_backbone(pretrained=True):
    """Create EfficientNet-B0 with robust pretrained weight loading.
    Falls back gracefully when HuggingFace is unreachable (Kaggle offline mode).
    """
    model_name = 'tf_efficientnet_b0_ns'
    try:
        return timm.create_model(model_name, pretrained=pretrained,
                                 in_chans=1, num_classes=0, global_pool='avg')
    except (RuntimeError, OSError, Exception) as e:
        if not pretrained:
            raise
        print(f'  ⚠ Online pretrained download failed: {type(e).__name__}: {e}')
        # Try 1: HuggingFace offline cache
        try:
            import os
            os.environ['HF_HUB_OFFLINE'] = '1'
            print('  → Trying HuggingFace local cache (HF_HUB_OFFLINE=1)...')
            bb = timm.create_model(model_name, pretrained=True,
                                   in_chans=1, num_classes=0, global_pool='avg')
            print('  ✓ Loaded from HuggingFace cache')
            return bb
        except Exception:
            pass
        # Try 2: Kaggle dataset path (common convention)
        for search_dir in ['/kaggle/input', '/kaggle/working']:
            import glob as _g
            for f in _g.glob(f'{search_dir}/**/tf_efficientnet_b0*.pth', recursive=True):
                print(f'  → Found local weights: {f}')
                bb = timm.create_model(model_name, pretrained=False,
                                       in_chans=1, num_classes=0, global_pool='avg')
                sd = torch.load(f, map_location='cpu', weights_only=True)
                bb.load_state_dict(sd, strict=False)
                print(f'  ✓ Loaded from local file')
                return bb
        # Final fallback: random init
        print('  ✗ No pretrained weights available. Using random initialization.')
        print('  ⚠ IMPORTANT: Enable Internet in Kaggle Settings → Internet → On')
        return timm.create_model(model_name, pretrained=False,
                                 in_chans=1, num_classes=0, global_pool='avg')


class BirdCLEFB0(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = _create_backbone(pretrained=pretrained)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(1280, CFG.NUM_CLASSES)

    def forward(self, mel):
        x = self.backbone(mel)
        x = self.dropout(x)
        return self.fc(x)


_m = BirdCLEFB0(pretrained=False)
_out = _m(torch.randn(2, 1, 128, 500))
print(f'Model output: {_out.shape}, {sum(p.numel() for p in _m.parameters())/1e6:.1f}M params')
del _m, _out

In [ ]:
def _amp():
    return autocast(AMP_DEVICE) if AMP_DEVICE else autocast()

def train_ep(model, loader, crit, opt, scaler, ep_num=0):
    model.train(); tl, n = 0.0, 0
    for bi, batch in enumerate(tqdm(loader, leave=False)):
        mel, tgt, w = batch[0], batch[1], batch[2] if len(batch) > 2 else None
        mel = mel.to(DEVICE, non_blocking=True)
        tgt = tgt.to(DEVICE, non_blocking=True)
        sw = w.to(DEVICE).view(-1) if w is not None else torch.ones(mel.size(0), device=DEVICE)
        if np.random.rand() < CFG.MIXUP_P: mel, tgt = mixup(mel, tgt)
        opt.zero_grad(set_to_none=True)
        if scaler:
            with _amp():
                logits = model(mel)
                loss_mat = crit(logits, tgt)
                loss_per_sample = loss_mat.mean(dim=1)
                loss = (loss_per_sample * sw).sum() / (sw.sum() + 1e-8)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            logits = model(mel)
            loss_mat = crit(logits, tgt)
            loss_per_sample = loss_mat.mean(dim=1)
            loss = (loss_per_sample * sw).sum() / (sw.sum() + 1e-8)
            loss.backward(); opt.step()
        tl += loss.item()*mel.size(0); n += mel.size(0)
    return tl/n

@torch.no_grad()
def val(model, loader, ep_num=0):
    model.eval(); ps, ts = [], []
    for bi, batch in enumerate(tqdm(loader, leave=False)):
        mel = batch[0].to(DEVICE, non_blocking=True)
        tgt = batch[1]
        if USE_AMP:
            with _amp(): logits = model(mel)
        else: logits = model(mel)
        probs = torch.sigmoid(logits).cpu().numpy()
        ps.append(probs); ts.append(tgt.numpy())

    ap, at = np.concatenate(ps), np.concatenate(ts)
    at_bin = (at > 0.5).astype(np.float32)
    if ep_num in (0, 4, 9):
        print(f'  [DIAG ep{ep_num}] Pred stats: mean={ap.mean():.6f} max={ap.max():.4f}')
        pos_mask = at_bin > 0.5
        if pos_mask.any():
            print(f'    Pred@positive: mean={ap[pos_mask].mean():.6f}')

    v = [i for i in range(at_bin.shape[1]) if 0 < at_bin[:,i].sum() < len(at_bin)]
    if not v: return (0.0, 0)
    try:
        auc = roc_auc_score(at_bin[:,v], ap[:,v], average='macro')
    except Exception as e:
        print(f'  AUC error: {e}'); auc = 0.0
    return (auc, len(v))

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
folds = list(sgkf.split(train_df, train_df['primary_label'].astype(str), train_df['author'].astype(str)))

results = []
for fi in CFG.TRAIN_FOLDS:
    ti, vi = folds[fi]
    print(f'\n{"="*60}')
    print(f'  Fold {fi+1}/{CFG.N_FOLDS} | Train: {len(ti)}, Val: {len(vi)}')
    print(f'  Soundscape data: {len(sc_labels_df) if sc_labels_df is not None else 0} rows mixed at 30%')
    print(f'{"="*60}')
    tl = DataLoader(BirdDataset(train_df.iloc[ti], True, sc_ratio=0.3, bg_noise_p=0.0),
                    batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS,
                    pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=3)
    vl = DataLoader(BirdDataset(train_df.iloc[vi], False, sc_ratio=0.0, bg_noise_p=0.0),
                    batch_size=CFG.BATCH_SIZE*2, shuffle=False, num_workers=CFG.NUM_WORKERS,
                    pin_memory=True, persistent_workers=True, prefetch_factor=3)

    model = BirdCLEFB0(pretrained=True).to(DEVICE)
    crit = nn.BCEWithLogitsLoss(reduction='none')
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.EPOCHS, eta_min=1e-5)
    scaler = GradScaler() if USE_AMP else None
    best = 0.0

    for ep in range(CFG.EPOCHS):
        t0 = time.time()
        loss = train_ep(model, tl, crit, opt, scaler, ep_num=ep)
        auc, nc = val(model, vl, ep_num=ep); sched.step()
        tag = ''
        if auc > best:
            best = auc
            torch.save(model.state_dict(), CFG.OUTPUT_DIR/f'best_fold{fi}.pth')
            tag = ' *BEST'
        elapsed = time.time()-t0
        print(f'  Ep {ep+1:02d}/{CFG.EPOCHS} | loss={loss:.4f} auc={auc:.4f} ({nc}sp) | lr={opt.param_groups[0]["lr"]:.6f} | {elapsed:.0f}s{tag}')

    results.append(best)
    print(f'  Fold {fi+1} Best AUC: {best:.4f}')
    del model, opt, sched, scaler, crit; torch.cuda.empty_cache()

print(f'\nCV Mean AUC: {np.mean(results):.4f}')

In [ ]:
import json
meta = {'species_cols': SPECIES_COLS, 'label2idx': LABEL2IDX, 'tax_map': TAX_MAP, 'results': results}
with open(CFG.OUTPUT_DIR / 'train_meta.json', 'w') as f:
    json.dump(meta, f)
for p in sorted(CFG.OUTPUT_DIR.glob('*')):
    print(f'  {p.name} ({p.stat().st_size/1024:.0f}KB)')